[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [4]:
import torch
import torch.nn as nn
import math

In [5]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # pass  # W_q, W_k, W_v, W_o
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv):
        # pass  # Q from x_q, K/V from x_kv, no causal mask
        B, seq_q, _ = x_q.shape
        B, seq_k, _ = x_kv.shape
        q = self.W_q(x_q).view(B, seq_q, self.num_heads, -1).transpose(1, 2)
        k = self.W_k(x_kv).view(B, seq_k, self.num_heads, -1).transpose(1, 2)
        v = self.W_v(x_kv).view(B, seq_k, self.num_heads, -1).transpose(1, 2)

        score = q @ k.transpose(-1, -2) / math.sqrt(self.d_k)
        weight = torch.softmax(score, dim=-1)
        atten = weight @ v
        out = self.W_o(atten.transpose(1, 2).reshape(B, seq_q, -1))
        return out

In [6]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [7]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.6ms)
  ✅ [2/4] Q and KV different lengths (1.2ms)
  ✅ [3/4] No causal mask — all KV affects all Q (40.1ms)
  ✅ [4/4] Gradient flow (69.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (113.5ms total)
  Progress saved. Run status() to see your dashboard.

